# Basic Decision Tree from Scratch - Regression (Median)

In this notebook, we are going to build a decision tree from scratch using median as a threshold/split point. 

**We use median splits to focus on understanding how trees partition data for regression problem. Later we replace this with variance or MSE to make splits data-driven instead of structure-driven.**

## Dataset

Dataset: This is a makeup dataset that describe how much study hours a student put in and what exam score they can achieve.

Dataset 
| Study Hours  | Exam Score |
| ----- | ----- |
| 1    | 10    |
| 2    | 20    |
| 3    | 30    |
| 4    | 45    |
| 5    | 50   |
| 6    | 55   |
| 7    | 75   |
| 8    | 80   |
| 9    | 80   |
| 10    | 90   |
| 11    | 95   |
| 12    | 93   |
| 13    | 90   |
| 14    | 91   |
| 15    | 88   |

In [17]:
import pandas as pd
import json

In [18]:
df = pd.DataFrame({
    'studied': [1,2,3,4,5,6,7,8,9,10,11,12,13,14,15],
    'score':  [10,20,30,45,50,55,75,80,80,90,95,93,90,91,88]
})

In [19]:
df

,studied,score
0,1,10
1,2,20
2,3,30
3,4,45
4,5,50
5,6,55
6,7,75
7,8,80
8,9,80
9,10,90


## Tree Building Algorithm (Step by Step)

We are going to create a tree by using median as a cutoff. The steps are as follows:

1. Compute median of feature

2. Split data into:
      - left_node  where values <= median
      - right_node where values > median

3. Condition for stopping split: 
      - If the dataset is empty
      - If there is only one row left in the subset, no more split. 
      - If the variance is zero.

4. Continue splitting for non pure node until condition on step 3 satisfied.

### Step 1

In [20]:
median_studied = df['studied'].median()
print(f"Median studied: {median_studied}")

left_subset = df[df['studied'] <= median_studied]
right_subset = df[df['studied'] > median_studied]

print(f"Left subset:\n{left_subset}\n")
print(f"Left Subset Score Variance: {left_subset['score'].var()}")
print('---')
print(f"Right subset:\n{right_subset}\n")
print(f"Reft Subset Score Variance: {right_subset['score'].var()}")

Median studied: 8.0
Left subset:
   studied  score
0        1     10
1        2     20
2        3     30
3        4     45
4        5     50
5        6     55
6        7     75
7        8     80

Left Subset Score Variance: 617.4107142857143
---
Right subset:
    studied  score
8         9     80
9        10     90
10       11     95
11       12     93
12       13     90
13       14     91
14       15     88

Reft Subset Score Variance: 22.95238095238096


- Right node variance is NOT zero -> need to split further 
- Left node -> (Skip left node for manual trial)

### Step 2

In [21]:
manual_df = right_subset
median_studied = manual_df['studied'].median()
print(f"Median studied: {median_studied}")

left_subset = manual_df[manual_df['studied'] <= median_studied]
right_subset = manual_df[manual_df['studied'] > median_studied]

print(f"Left subset:\n{left_subset}\n")
print(f"Left Subset Score Variance: {left_subset['score'].var()}")
print('---')
print(f"Right subset:\n{right_subset}\n")
print(f"Reft Subset Score Variance: {right_subset['score'].var()}")

Median studied: 12.0
Left subset:
    studied  score
8         9     80
9        10     90
10       11     95
11       12     93

Left Subset Score Variance: 44.333333333333336
---
Right subset:
    studied  score
12       13     90
13       14     91
14       15     88

Reft Subset Score Variance: 2.3333333333333335


- Right node variance is NOT zero -> need to split further 
- Left node -> (Skip left node for manual trial)

### Step 3

In [22]:
manual_df = right_subset
median_studied = manual_df['studied'].median()
print(f"Median studied: {median_studied}")

left_subset = manual_df[manual_df['studied'] <= median_studied]
right_subset = manual_df[manual_df['studied'] > median_studied]

print(f"Left subset:\n{left_subset}\n")
print(f"Left Subset Score Variance: {left_subset['score'].var()}")
print('---')
print(f"Right subset:\n{right_subset}\n")
print(f"Reft Subset Score Variance: {right_subset['score'].var()}")

Median studied: 14.0
Left subset:
    studied  score
12       13     90
13       14     91

Left Subset Score Variance: 0.5
---
Right subset:
    studied  score
14       15     88

Reft Subset Score Variance: nan


- Right node Only ONE row left -> stop 
- Left node -> (Skip left node for manual trial)

### Create Tree Building Function

We are going to create a tree by using median as a cutoff. The steps are as follows:

Build(node_data):

1. Stopping condition:
      - If the dataset is empty:
            - return None
      - If there is only one row left:
            - create leaf and return prediction
      - If the variance is zero:
            - create leaf and return prediction

2. Compute median of feature

3. Split rows into:
      - left_node  where values <= median
      - right_node where values > median

4. Create node storing the median threshold

5. Build(left) Recursively

6. Build(right) Recursively

In [23]:
def build_tree_regression(df):

    # ---- STOP 1: empty node ----
    if df.empty:
        return None

    # ---- STOP 2: too small to split ----
    if len(df) <= 1:
        print("Too small to split, one row left, returning mean score")
        return float(df['score'].mean())
    
    # ---- STOP 3: variance is zero ----
    if df['score'].var() == 0:
        print("Variance is zero, returning mean score")
        return float(df['score'].mean())

    # ---- compute median split ----
    median_threshold = df['studied'].median()

    print(f"Median studied: {median_threshold}")

    # ---- split data ----
    left_subset = df[df['studied'] <= median_threshold]
    right_subset = df[df['studied'] > median_threshold]

    print(f"Left subset:\n{left_subset}\n")
    print(f"Right subset:\n{right_subset}\n")
    
    # ---- build node ----
    node = {
        'median_threshold': float(median_threshold),
        'value': round(float(df['score'].mean()), 2),  # 👈 REGRESSION CHANGE
        'left': None,
        'right': None
    }

    # ---- recursion ----
    node['left'] = build_tree_regression(left_subset)
    node['right'] = build_tree_regression(right_subset)

    return node

### Build Tree 

We build a tree using current data frame.

In [24]:
df

,studied,score
0,1,10
1,2,20
2,3,30
3,4,45
4,5,50
5,6,55
6,7,75
7,8,80
8,9,80
9,10,90


In [25]:
reg = build_tree_regression(df)

Median studied: 8.0
Left subset:
   studied  score
0        1     10
1        2     20
2        3     30
3        4     45
4        5     50
5        6     55
6        7     75
7        8     80

Right subset:
    studied  score
8         9     80
9        10     90
10       11     95
11       12     93
12       13     90
13       14     91
14       15     88

Median studied: 4.5
Left subset:
   studied  score
0        1     10
1        2     20
2        3     30
3        4     45

Right subset:
   studied  score
4        5     50
5        6     55
6        7     75
7        8     80

Median studied: 2.5
Left subset:
   studied  score
0        1     10
1        2     20

Right subset:
   studied  score
2        3     30
3        4     45

Median studied: 1.5
Left subset:
   studied  score
0        1     10

Right subset:
   studied  score
1        2     20

Too small to split, one row left, returning mean score
Too small to split, one row left, returning mean score
Median studied: 3.5


In [26]:
reg


{'median_threshold': 8.0,
 'value': 66.13,
 'left': {'median_threshold': 4.5,
  'value': 45.62,
  'left': {'median_threshold': 2.5,
   'value': 26.25,
   'left': {'median_threshold': 1.5,
    'value': 15.0,
    'left': 10.0,
    'right': 20.0},
   'right': {'median_threshold': 3.5,
    'value': 37.5,
    'left': 30.0,
    'right': 45.0}},
  'right': {'median_threshold': 6.5,
   'value': 65.0,
   'left': {'median_threshold': 5.5,
    'value': 52.5,
    'left': 50.0,
    'right': 55.0},
   'right': {'median_threshold': 7.5,
    'value': 77.5,
    'left': 75.0,
    'right': 80.0}}},
 'right': {'median_threshold': 12.0,
  'value': 89.57,
  'left': {'median_threshold': 10.5,
   'value': 89.5,
   'left': {'median_threshold': 9.5,
    'value': 85.0,
    'left': 80.0,
    'right': 90.0},
   'right': {'median_threshold': 11.5,
    'value': 94.0,
    'left': 95.0,
    'right': 93.0}},
  'right': {'median_threshold': 14.0,
   'value': 89.67,
   'left': {'median_threshold': 13.5,
    'value': 90.5

In [27]:
# print tree structure in json format
print(json.dumps(reg, indent=8))


{
        "median_threshold": 8.0,
        "value": 66.13,
        "left": {
                "median_threshold": 4.5,
                "value": 45.62,
                "left": {
                        "median_threshold": 2.5,
                        "value": 26.25,
                        "left": {
                                "median_threshold": 1.5,
                                "value": 15.0,
                                "left": 10.0,
                                "right": 20.0
                        },
                        "right": {
                                "median_threshold": 3.5,
                                "value": 37.5,
                                "left": 30.0,
                                "right": 45.0
                        }
                },
                "right": {
                        "median_threshold": 6.5,
                        "value": 65.0,
                        "left": {
                                "median_threshold": 5

In [28]:
# print tree structure with guided lines
def print_tree(node, indent="    "):
    if isinstance(node, dict):
        print(f"{indent}Median Threshold: {node['median_threshold']}")
        print(f"{indent}├─ Left Node: <= {node['median_threshold']}")
        print_tree(node['left'], indent + "│   ")
        print(f"{indent}└─ Right Node: > {node['median_threshold']}")
        print_tree(node['right'], indent + "    ")
    else:
        print(f"{indent} ->Score: {node}")

In [29]:
print_tree(reg)

    Median Threshold: 8.0
    ├─ Left Node: <= 8.0
    │   Median Threshold: 4.5
    │   ├─ Left Node: <= 4.5
    │   │   Median Threshold: 2.5
    │   │   ├─ Left Node: <= 2.5
    │   │   │   Median Threshold: 1.5
    │   │   │   ├─ Left Node: <= 1.5
    │   │   │   │    ->Score: 10.0
    │   │   │   └─ Right Node: > 1.5
    │   │   │        ->Score: 20.0
    │   │   └─ Right Node: > 2.5
    │   │       Median Threshold: 3.5
    │   │       ├─ Left Node: <= 3.5
    │   │       │    ->Score: 30.0
    │   │       └─ Right Node: > 3.5
    │   │            ->Score: 45.0
    │   └─ Right Node: > 4.5
    │       Median Threshold: 6.5
    │       ├─ Left Node: <= 6.5
    │       │   Median Threshold: 5.5
    │       │   ├─ Left Node: <= 5.5
    │       │   │    ->Score: 50.0
    │       │   └─ Right Node: > 5.5
    │       │        ->Score: 55.0
    │       └─ Right Node: > 6.5
    │           Median Threshold: 7.5
    │           ├─ Left Node: <= 7.5
    │           │    ->Score: 75.0
    │

### Prediction / Inference

To make a prediction we pass the prediction to the tree until we reach a leaf.

In [30]:
# make prediction function
def predict(reg, x):
    node = reg  # start at root of the tree

    # while we are still at a decision node (dictionary)
    while isinstance(node, dict):

        # get the split threshold stored at this node
        threshold = node['median_threshold']

        # go left if x is smaller or equal to threshold
        if x <= threshold:
            print(f"Going left: {x} <= {threshold}")
            node = node['left']
        # otherwise go right
        else:
            print(f"Going right: {x} > {threshold}")
            node = node['right']

    # when we reach a leaf (not a dict), return the prediction
    return node

In [31]:
number_of_hours_studied = 9

prediction = predict(reg, number_of_hours_studied)
print(f"Prediction for student who studied {number_of_hours_studied} hours: Estimated score: {prediction}")

Going right: 9 > 8.0
Going left: 9 <= 12.0
Going left: 9 <= 10.5
Going left: 9 <= 9.5
Prediction for student who studied 9 hours: Estimated score: 80.0


### Why any values above 16 always return 88

If we traverse the tree, it will eventually reach the right most leaf which give 88. Values above 15 return 88 because decision tree routes all unseen large inputs into the same rightmost leaf region, and that leaf stores the mean of training targets in that region. A decision tree is NOT 'interpolating values', it is 'routing inputs into regions'. Decision trees cannot extrapolate outside training range

This is the limitations of trees:

- they give step-like predictions
- there is no smooth extrapolation

We use random forests, gradient boosting and linear models in leaves to overcome the limitations.

## End